# MPRF — α imbriqué et robustesse sur 5 seeds
Placer les 4 fichiers de données dans `/content` (mêmes noms que pour les cellules précédentes), puis exécuter la cellule. Sorties : `mprf_nested5_raw.csv`, `mprf_robustness5_raw.csv`.

In [ ]:
# ============================================================
# MPRF — Deux contrôles sur 5 seeds (au lieu du seed 0 seul)
#  A. alpha choisi par validation imbriquée (CV interne à 3 plis sur le train de chaque pli) vs alpha=0.6 fixe,
#     sur les 4 datasets, 5 seeds
#  B. Cervical : imputation de la médiane dans les plis, puis sans Hinselmann/Schiller/Citology (32 variables),
#     puis sans ces 3 variables ni les Dx (28 variables) — CSRF, OBP, IBEP, MPRF, 5 seeds
# Même protocole que cellD (forêt random_state=42+seed ; validation RandomState(pli+1000*seed)).
# Aucun fichier du papier n'est modifié. Google Colab — copier-coller directement
# ============================================================
import os, time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings("ignore")

# ┌──────────────────────── PARAMÈTRES ────────────────────────┐
DATASETS = ["cervical", "hepatitis", "pima", "mammography"]   # retirer ceux que tu ne veux pas relancer
SEEDS    = [0, 1, 2, 3, 4]          # seed 0 = exactement le protocole du papier
ALPHAS   = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
KS       = [5, 10, 15, 20]
ALPHA0, K0 = 0.6, 10                # configuration du papier
N_TREES, N_SPLITS = 50, 10
NESTED, NESTED_SEEDS, INNER_SPLITS = True, [0], 3   # alpha choisi par CV interne (3 plis) sur le train
N_BOOT   = 2000
OUT_DIR  = "."                      # les CSV sont écrits ici
FILES = {                           # mêmes fichiers que dans ton notebook
    "cervical":    "1. risk_factors_cervical_cancer.csv",
    "hepatitis":   "2. HepatitisCdata.csv",
    "pima":        "3. diabetes.csv",
    "mammography": "/content/phpn1jVwe.arff",
}
# └─────────────────────────────────────────────────────────────┘


# ┌──────────────────────── PARAMÈTRES ────────────────────────┐
DATASETS = ["cervical", "hepatitis", "pima", "mammography"]
SEEDS    = [0, 1, 2, 3, 4]
ALPHAS   = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
ALPHA0, K0 = 0.6, 10
N_TREES, N_SPLITS, INNER_SPLITS, N_BOOT = 50, 10, 3, 2000
OUT_DIR  = "."
FILES = {
    "cervical":    "1. risk_factors_cervical_cancer.csv",
    "hepatitis":   "2. HepatitisCdata.csv",
    "pima":        "3. diabetes.csv",
    "mammography": "/content/phpn1jVwe.arff",
}
EXAMS = ["Hinselmann", "Schiller", "Citology"]
DX    = ["Dx:Cancer", "Dx:CIN", "Dx:HPV", "Dx"]
# └─────────────────────────────────────────────────────────────┘

# ── Chargement (identique à ton notebook) ───────────────────
def load(name):
    path = FILES[name]
    if not os.path.exists(path):
        print(f"⚠️  {name}: fichier introuvable ({path}) → ignoré"); return None
    if name == "cervical":
        df = pd.read_csv(path); df.replace("?", np.nan, inplace=True)
        df = df.apply(pd.to_numeric, errors="coerce"); df.fillna(df.median(numeric_only=True), inplace=True)
        feats = [c for c in df.columns if c != "Biopsy"]
        return df[feats].values, df["Biopsy"].values.astype(int)
    if name == "hepatitis":
        df = pd.read_csv(path); df["Category"] = df["Category"].astype(str).str.strip()
        df["Category"] = df["Category"].map(lambda x: 0 if "Blood Donor" in x else (1 if "Hepatitis" in x else np.nan))
        df = df.dropna(subset=["Category"])
        feats = [c for c in df.columns if c not in ["Category", "Unnamed: 0"]]
        d = df[feats].copy()
        for col in d.columns:
            if d[col].dtype == object:
                le = LabelEncoder(); m = d[col].notna()
                d.loc[m, col] = le.fit_transform(d.loc[m, col].astype(str)); d[col] = pd.to_numeric(d[col], errors="coerce")
        d = d.apply(pd.to_numeric, errors="coerce"); d.fillna(d.median(numeric_only=True), inplace=True)
        return d.values, df["Category"].values.astype(int)
    if name == "pima":
        df = pd.read_csv(path); df.fillna(df.median(numeric_only=True), inplace=True)
        return df[[c for c in df.columns if c != "Outcome"]].values, df["Outcome"].values.astype(int)
    if name == "mammography":
        rows, data = [], False
        for line in open(path):
            line = line.strip()
            if line.lower() == "@data": data = True; continue
            if data and line:
                v = line.split(","); rows.append([float(t) for t in v[:6]] + [1 if v[6].strip().strip("'") == "1" else 0])
        a = np.array(rows); return a[:, :6], a[:, 6].astype(int)

# ── Métriques / score (mêmes formules que le notebook) ──────
def counts(y, p):
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum()); tn = int(((y == 0) & (p == 0)).sum())
    return tp, fp, fn, tn

def f1_gm(tp, fp, fn, tn):
    den = 2 * tp + fp + fn
    f1 = 2 * tp / den if den > 0 else 0.0
    r1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    r0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return f1, np.sqrt(r1 * r0)

def metrics(y, p, proba):
    tp, fp, fn, tn = counts(y, p); f1, gm = f1_gm(tp, fp, fn, tn)
    return dict(f1_min=f1, gmean=gm,
                precision=tp / (tp + fp) if (tp + fp) > 0 else 0.0,
                sens=tp / (tp + fn) if (tp + fn) > 0 else 0.0,
                spec=tn / (tn + fp) if (tn + fp) > 0 else 0.0,
                fp=fp, fn=fn, auprc=average_precision_score(y, proba))

def score_vec(yv, pred_mat, alpha):          # pred_mat : (candidats, n_val) booléens
    pos = (yv == 1); neg = ~pos
    tp = (pred_mat & pos).sum(1); fp = (pred_mat & neg).sum(1)
    fn = (~pred_mat & pos).sum(1); tn = (~pred_mat & neg).sum(1)
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    r1 = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    r0 = np.where(tn + fp > 0, tn / np.maximum(tn + fp, 1), 0.0)
    return alpha * f1 + (1 - alpha) * np.sqrt(r1 * r0)

# ── Sélection d'arbres (P = matrice (arbres, instances) des prédictions 0/1) ──
def MPRF(Pv, yv, k, alpha, init_seed, init="random"):
    n = Pv.shape[0]
    if init == "random":
        i0 = np.random.RandomState(init_seed).randint(0, n)
    else:                                         # "best" : meilleur arbre seul sur la validation
        i0 = int(np.argmax(score_vec(yv, Pv.astype(bool), alpha)))
    sel = [i0]; rem = [i for i in range(n) if i != i0]
    votes = Pv[i0].astype(float).copy()
    best = score_vec(yv, (votes >= 0.5)[None, :], alpha)[0]
    while len(sel) < k and rem:
        V = votes[None, :] + Pv[rem]
        sc = score_vec(yv, V >= (len(sel) + 1) / 2, alpha)
        j = int(np.argmax(sc))                    # 1er maximum = même règle que la boucle d'origine
        if sc[j] >= best:                         # plateau accepté
            votes = V[j]; sel.append(rem.pop(j))
            if sc[j] > best: best = sc[j]
        else:
            break
    return sel

def OBP(Pv, yv, max_size):
    n = Pv.shape[0]
    sc = [f1_gm(*counts(yv, Pv[i]))[0] for i in range(n)]
    order = np.argsort(sc)[::-1]; sel, best, votes = [], -np.inf, np.zeros(Pv.shape[1])
    for idx in order[:max_size]:
        v = votes + Pv[idx]; s = f1_gm(*counts(yv, (v >= (len(sel) + 1) / 2).astype(int)))[0]
        if s >= best: sel.append(idx); votes = v; best = s
    return sel if sel else [order[0]]

def IBEP(Pv, yv, max_size):
    n = Pv.shape[0]; err = (Pv != yv).astype(float); w = np.ones(len(yv)) / len(yv); sel = []
    for _ in range(max_size):
        b = int(np.argmin(err.dot(w)))
        if b in sel: break
        sel.append(b); w *= np.exp(err[b]); w /= w.sum()
    return sel if sel else [0]

def vote(P, sel):  return (P[sel].sum(0) >= len(sel) / 2).astype(int)

def forest(Xa, ya, seed, balanced):
    return RandomForestClassifier(n_estimators=N_TREES, class_weight="balanced" if balanced else None,
                                  random_state=42 + seed, n_jobs=-1).fit(Xa, ya)

def tree_mats(rf, Xv, Xt):
    Pv = np.array([t.predict(Xv) for t in rf.estimators_])
    Pt = np.array([t.predict(Xt) for t in rf.estimators_])
    Qt = np.array([t.predict_proba(Xt)[:, 1] for t in rf.estimators_])
    return Pv, Pt, Qt

# ── alpha par validation imbriquée (n'utilise que le train du pli externe) ──
def nested_alpha(Xtr, ytr, fold, seed):
    skf = StratifiedKFold(INNER_SPLITS, shuffle=True, random_state=seed + 7)
    acc = {a: [] for a in ALPHAS}
    for j, (itr, ite) in enumerate(skf.split(Xtr, ytr)):
        rng = np.random.RandomState(fold * 31 + j + 1000 * seed)
        vi = rng.choice(len(itr), int(len(itr) * 0.2), replace=False); mask = np.ones(len(itr), bool); mask[vi] = False
        rf = forest(Xtr[itr][mask], ytr[itr][mask], seed, True)
        Pv, Pt, _ = tree_mats(rf, Xtr[itr][vi], Xtr[ite])
        for a in ALPHAS:
            sel = MPRF(Pv, ytr[itr][vi], K0, a, 100 + fold + 1000 * seed)
            acc[a].append(f1_gm(*counts(ytr[ite], vote(Pt, sel)))[0])
    m = {a: np.mean(v) for a, v in acc.items()}
    return max(ALPHAS, key=lambda a: (m[a], -abs(a - ALPHA0)))

rng_b = np.random.RandomState(0)
def boot_ci(v):
    v = np.asarray(v, float); m = [rng_b.choice(v, len(v)).mean() for _ in range(N_BOOT)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

# ── A. alpha imbriqué sur 5 seeds ───────────────────────────
def run_nested(name, X, y):
    rows = []; skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=42)
    for seed in SEEDS:
        for fold, (tr, te) in enumerate(skf.split(X, y)):
            Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]
            vi = np.random.RandomState(fold + 1000 * seed).choice(len(ytr), int(len(tr) * 0.2), replace=False)
            sub = np.setdiff1d(np.arange(len(ytr)), vi)
            Xv, yv, Xs, ys = Xtr[vi], ytr[vi], Xtr[sub], ytr[sub]
            iseed = 100 + fold + 1000 * seed
            rf = forest(Xs, ys, seed, True); Pv, Pt, Qt = tree_mats(rf, Xv, Xte)
            a_star = nested_alpha(Xtr, ytr, fold, seed)
            for nm, a in (("fixe a=0.6", ALPHA0), ("imbriqué", a_star)):
                sel = MPRF(Pv, yv, K0, a, iseed)
                d = metrics(yte, vote(Pt, sel), Qt[sel].mean(0)); d.update(dataset=name, seed=seed, fold=fold, method=nm, size=len(sel), alpha=a); rows.append(d)
        print(f"   {name}: seed {seed} terminé", flush=True)
    return pd.DataFrame(rows)

def report_nested(R, name):
    D = R[R.dataset == name]; per = D.groupby(["method", "seed"]).mean(numeric_only=True).reset_index()
    fold = D.groupby(["method", "fold"]).mean(numeric_only=True).reset_index()
    print("\n" + "=" * 96); print(f"  {name.upper()} — alpha imbriqué vs alpha=0.6 fixe (5 seeds ; ± = écart-type entre seeds)"); print("=" * 96)
    print(f"   {'méthode':<12} {'taille':>6} {'F1':>13} {'G-mean':>13} {'AUPRC':>13}")
    for m in ["fixe a=0.6", "imbriqué"]:
        p = per[per.method == m]; s = lambda c: f"{p[c].mean():.3f}±{p[c].std(ddof=0):.3f}"
        print(f"   {m:<12} {p['size'].mean():6.1f} {s('f1_min'):>13} {s('gmean'):>13} {s('auprc'):>13}")
    print("   F1 par seed (fixe / imbriqué) :", "  ".join(f"s{s}: {per[(per.method=='fixe a=0.6')&(per.seed==s)].f1_min.values[0]:.3f}/{per[(per.method=='imbriqué')&(per.seed==s)].f1_min.values[0]:.3f}" for s in SEEDS))
    for met in ["f1_min", "gmean"]:
        x = fold[fold.method == "imbriqué"].sort_values("fold")[met].values; y_ = fold[fold.method == "fixe a=0.6"].sort_values("fold")[met].values
        dd = x - y_; lo, hi = boot_ci(dd)
        try: p = 1.0 if np.all(dd == 0) else wilcoxon(x, y_)[1]
        except Exception: p = np.nan
        print(f"   imbriqué − fixe, {met:<7}: Δ={dd.mean():+.3f} [{lo:+.3f},{hi:+.3f}]  p brut={p:.4f}")
    al = D[D.method == "imbriqué"].alpha.round(1).value_counts().sort_index()
    print("   alpha choisi (50 choix = 10 plis x 5 seeds) :", {float(k): int(v) for k, v in al.items()})

# ── B. Cervical : imputation dans les plis, sans variables d'examen ──
def run_rob(drop, label):
    df = pd.read_csv(FILES["cervical"]); df.replace("?", np.nan, inplace=True); df = df.apply(pd.to_numeric, errors="coerce")
    feats = [c for c in df.columns if c != "Biopsy" and c not in drop]
    X = df[feats].values.astype(float); y = df["Biopsy"].values.astype(int)
    rows = []; skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=42)
    for seed in SEEDS:
        for fold, (tr, te) in enumerate(skf.split(X, y)):
            med = np.nanmedian(X[tr], axis=0); med = np.where(np.isnan(med), 0, med)       # médiane du train uniquement
            Xtr = np.where(np.isnan(X[tr]), med, X[tr]); Xte = np.where(np.isnan(X[te]), med, X[te]); ytr, yte = y[tr], y[te]
            vi = np.random.RandomState(fold + 1000 * seed).choice(len(ytr), int(len(tr) * 0.2), replace=False)
            sub = np.setdiff1d(np.arange(len(ytr)), vi)
            Xv, yv, Xs, ys = Xtr[vi], ytr[vi], Xtr[sub], ytr[sub]; iseed = 100 + fold + 1000 * seed
            rf = forest(Xs, ys, seed, True); Pv, Pt, Qt = tree_mats(rf, Xv, Xte)
            def add(m, pred, proba, size):
                d = metrics(yte, pred, proba); d.update(setting=label, seed=seed, fold=fold, method=m, size=size); rows.append(d)
            add("CSRF", rf.predict(Xte), rf.predict_proba(Xte)[:, 1], N_TREES)
            for nm, sel in (("OBP", OBP(Pv, yv, N_TREES)), ("IBEP", IBEP(Pv, yv, N_TREES)), ("MPRF", MPRF(Pv, yv, K0, ALPHA0, iseed))):
                add(nm, vote(Pt, sel), Qt[sel].mean(0), len(sel))
    return pd.DataFrame(rows), len(feats)

def report_rob(R, n):
    per = R.groupby(["method", "seed"]).mean(numeric_only=True).reset_index(); fold = R.groupby(["method", "fold"]).mean(numeric_only=True).reset_index()
    print("\n" + "=" * 96); print(f"  CERVICAL — {R.setting.iloc[0]}  [{n} variables] (5 seeds ; ± = écart-type entre seeds)"); print("=" * 96)
    print(f"   {'méthode':<6} {'taille':>6} {'F1':>13} {'G-mean':>13} {'AUPRC':>13} {'Préc':>6} {'Rappel':>7}")
    for m in ["CSRF", "MPRF", "OBP", "IBEP"]:
        p = per[per.method == m]; s = lambda c: f"{p[c].mean():.3f}±{p[c].std(ddof=0):.3f}"
        print(f"   {m:<6} {p['size'].mean():6.1f} {s('f1_min'):>13} {s('gmean'):>13} {s('auprc'):>13} {p['precision'].mean():6.3f} {p['sens'].mean():7.3f}")
    for other in ["CSRF", "OBP", "IBEP"]:
        x = fold[fold.method == "MPRF"].sort_values("fold").f1_min.values; y_ = fold[fold.method == other].sort_values("fold").f1_min.values
        dd = x - y_; lo, hi = boot_ci(dd)
        try: p = 1.0 if np.all(dd == 0) else wilcoxon(x, y_)[1]
        except Exception: p = np.nan
        print(f"   F1 MPRF − {other:<5}: Δ={dd.mean():+.3f} [{lo:+.3f},{hi:+.3f}]  p brut={p:.4f}")

t0 = time.time(); RN = []; RR = []
print("\n######## A. alpha imbriqué sur 5 seeds ########")
for name in DATASETS:
    ld = load(name)
    if ld is None: continue
    X, y = ld; c = np.bincount(y); print(f"\n### {name} : {X.shape} | classes {c.tolist()}")
    RN.append(run_nested(name, X, y)); report_nested(pd.concat(RN), name)
    pd.concat(RN).to_csv(os.path.join(OUT_DIR, "mprf_nested5_raw.csv"), index=False)
if os.path.exists(FILES["cervical"]):
    print("\n######## B. Cervical : imputation dans les plis / sans variables d'examen (5 seeds) ########")
    for drop, label in (([], "imputation dans les plis, 35 variables"), (EXAMS, "sans Hinselmann/Schiller/Citology"), (EXAMS + DX, "sans ces 3 variables ni les Dx")):
        R, n = run_rob(drop, label); RR.append(R); report_rob(R, n)
    pd.concat(RR).to_csv(os.path.join(OUT_DIR, "mprf_robustness5_raw.csv"), index=False)
print(f"\nTerminé en {(time.time()-t0)/60:.1f} min — mprf_nested5_raw.csv, mprf_robustness5_raw.csv")
